In [19]:
import math

def calculate_shot_angle_optimized(
    v0: float,
    distance: float,
    target_height: float = 1.8288,   # default hub height
    muzzle_height: float = 0.508,    # robot shooter height
    ball_properties = None,
    env_properties = None,
    use_drag: bool = True,
    arc: str = "low",                # "low", "high", or "both"
    dt: float = 0.01,                # INCREASED: 0.01 is safe with RK2
    coarse_step_deg: float = 1.0,    # INCREASED: Scan every 5 deg, not 0.5
    tol_m: float = 0.01,             # Acceptable error (1 cm)
    max_time_s: float = 5.0          # Reduced safety timeout for speed
    ):

    # --- 1) Defaults & Constants ---
    if ball_properties is None:
        ball_properties = {"mass": 0.215, "radius": 0.075, "drag_coeff": 0.5}
    if env_properties is None:
        env_properties = {"gravity": 9.81, "air_density": 1.225}

    g = env_properties["gravity"]
    rho = env_properties["air_density"]
    
    m = ball_properties["mass"]
    r = ball_properties["radius"]
    Cd = ball_properties["drag_coeff"]
    area = math.pi * r * r
    k = 0.5 * rho * Cd * area  # Drag factor: F = k * v^2

    dx = float(distance)
    dy = float(target_height - muzzle_height)

    # Sanity check
    if v0 <= 0 or dx <= 0:
        return (None, None) if arc == "both" else None

    # --- 2) Fast Vacuum Check (Analytic) ---
    # If we can't hit it in a vacuum, we definitely can't hit it with air.
    v_sq = v0 * v0
    term1 = (g * dx * dx) / (2.0 * v_sq)
    A = term1
    B = -dx
    C = dy + term1
    disc = B * B - 4.0 * A * C

    if disc < 0:
        return (None, None) if arc == "both" else None  # Unreachable

    # If drag is disabled, return vacuum solutions directly
    if not use_drag:
        sqrt_disc = math.sqrt(disc)
        tan_low  = (-B - sqrt_disc) / (2.0 * A)
        tan_high = (-B + sqrt_disc) / (2.0 * A)
        low = math.degrees(math.atan(tan_low))
        high = math.degrees(math.atan(tan_high))
        
        if arc == "low": return low
        if arc == "high": return high
        return (low, high)

    # --- 3) RK2 (Heun's Method) Flight Simulation ---
    # Faster and more stable than Euler. Allows larger dt.
    def simulate_y_at_dx(angle_rad: float):
        x, y = 0.0, 0.0
        vx = v0 * math.cos(angle_rad)
        vy = v0 * math.sin(angle_rad)

        floor_limit = -muzzle_height - 0.5 # Fail fast if we hit floor
        t = 0.0
        
        while t < max_time_s:
            if y < floor_limit:
                return None # Grounded
            
            # Current State
            v = math.hypot(vx, vy)
            if v < 1e-1: return None # Stopped

            # --- RK2 Step 1: Predictor (Euler) ---
            fd = k * v * v
            ax = -(fd * (vx / v)) / m
            ay = -g - (fd * (vy / v)) / m
            
            # Predict position at half-step
            half_dt = 0.5 * dt
            mid_vx = vx + ax * half_dt
            mid_vy = vy + ay * half_dt
            mid_v = math.hypot(mid_vx, mid_vy)

            # --- RK2 Step 2: Corrector ---
            mid_fd = k * mid_v * mid_v
            mid_ax = -(mid_fd * (mid_vx / mid_v)) / m
            mid_ay = -g - (mid_fd * (mid_vy / mid_v)) / m

            # Update using midpoint acceleration
            prev_x, prev_y = x, y
            vx += mid_ax * dt
            vy += mid_ay * dt
            x += vx * dt
            y += vy * dt
            t += dt

            # Check crossing
            if x >= dx:
                # Linear interpolation for sub-dt precision
                if x == prev_x: return y
                frac = (dx - prev_x) / (x - prev_x)
                return prev_y + frac * (y - prev_y)

        return None # Timeout

    def err(angle_rad):
        y_at = simulate_y_at_dx(angle_rad)
        if y_at is None: return None
        return y_at - dy

    # --- 4) Coarse Scan for Brackets ---
    def find_brackets():
        brackets = []
        # Integer-based loop to prevent float drift
        start_deg, end_deg = 1.0, 85.0
        steps = int((end_deg - start_deg) / coarse_step_deg)
        
        angles = [math.radians(start_deg + i * coarse_step_deg) for i in range(steps + 1)]

        prev_a = angles[0]
        prev_e = err(prev_a)

        for a in angles[1:]:
            e = err(a)
            
            # If we cross from valid to valid and change sign -> Bracket found
            if prev_e is not None and e is not None:
                if prev_e * e <= 0:
                    brackets.append((prev_a, a))
            
            # Handle edge case where solution is exactly on the scan point
            # (Rare, but good to catch)
            if e == 0.0:
                 brackets.append((a, a))

            prev_a, prev_e = a, e
        return brackets

    # --- 5) Bisection Solver ---
    def bisect(a, b):
        # Quick checks
        ea, eb = err(a), err(b)
        if ea is None or eb is None: return None
        if abs(ea) < tol_m: return a
        if abs(eb) < tol_m: return b
        
        lo, hi = a, b
        for _ in range(20): # 20 iterations is plenty for 5 degree gap
            mid = 0.5 * (lo + hi)
            em = err(mid)
            
            if em is None:
                # Midpoint failed (hit ground/ceiling). 
                # This is the risky case. Usually implies we are skirting the edge of range.
                # Safe bet: fail this bracket.
                return None 

            if abs(em) <= tol_m:
                return mid
            
            if err(lo) * em < 0:
                hi = mid
            else:
                lo = mid
        return 0.5 * (lo + hi)

    # --- Execution ---
    brackets = find_brackets()
    
    if not brackets:
        return (None, None) if arc == "both" else None

    # Resolve brackets
    low_sol_rad = bisect(*brackets[0])
    high_sol_rad = bisect(*brackets[-1])
    
    low_deg = math.degrees(low_sol_rad) if low_sol_rad else None
    high_deg = math.degrees(high_sol_rad) if high_sol_rad else None

    if arc == "low": return low_deg
    if arc == "high": return high_deg
    return (low_deg, high_deg)

In [26]:
test_v0 = 16.54
test_dist = 6.14

result = calculate_shot_angle_optimized(
    v0=test_v0,
    distance=test_dist,
    target_height=2.1,
    coarse_step_deg=1.0,
    arc="both"  # <--- Requesting tuple
)

print(f"Testing shot at {test_dist}m with {test_v0}m/s...")

if isinstance(result, tuple):
    print("\n[SUCCESS] Tuple returned (arc='both' used):")
    low, high = result
    
    # Format 'Low' (Direct shot)
    if low is not None:
        print(f"  • Low Angle:  {low:.2f}°")
    else:
        print(f"  • Low Angle:  None (Obstructed/Unreachable)")
        
    # Format 'High' (Lob shot)
    if high is not None:
        print(f"  • High Angle: {high:.2f}°")
    else:
        print(f"  • High Angle: None (Too high/Unreachable)")

else:
    # This happens if you change arc="low" or arc="high"
    print(f"\nSingle value returned: {result}")

Testing shot at 6.14m with 16.54m/s...

[SUCCESS] Tuple returned (arc='both' used):
  • Low Angle:  22.12°
  • High Angle: 79.84°


# Projectile Motion Solver: Step-by-Step Example

This walkthrough demonstrates how the `calculate_shot_angle_optimized` function finds both firing solutions for your specific inputs.

### 1. The Scenario

* **Muzzle Velocity ():**  m/s
* **Distance to Target ():**  meters
* **Target Height ():**  meters
* **Muzzle Height ():**  meters

**Target Relative Height ():** 

---

### 2. Step 1: The Vacuum Check (Sanity Test)

* **Formula:** Standard kinematic projectile equation.
* **Result:** In a vacuum, a  m/s shot can easily reach m height at m distance (Approx vacuum solution: ).
* *Status:* **PASSED**. Proceed to drag simulation.

---

### 3. Step 2: The Coarse Scan (Now 1.0° Steps)

The code scans through angles in ** increments**. This finer resolution allows it to catch the steep "lob" shot which has a very narrow window of validity.

**Scan Area 1: The Low Angle (Direct Shot)**
| Scan Angle | Error () | Conclusion |
| :--- | :--- | :--- |
|  |  m | Too Low (Under target) |
|  |  m | Too Low |
|  |  m | **Too High** (Over target) |

**🚩 BRACKET 1 FOUND!**
The error sign flipped between ** and **. The Low solution is in this window.

**Scan Area 2: The High Angle (Lob Shot)**
| Scan Angle | Error () | Conclusion |
| :--- | :--- | :--- |
|  |  m | Too High (Over target) |
|  |  m | **Too Low** (Falls short) |

**🚩 BRACKET 2 FOUND!**
The error sign flipped between ** and **. The High solution is in this window.

---

### 4. Step 3: Bisection (Drilling Down)

**Solving Bracket 1 ():**

1. Check Midpoint : **Too High**
2. Check Midpoint : **Perfect Hit** ()

**Final Low Result:** 

**Solving Bracket 2 ():**

1. Check Midpoint : **Too High**
2. Check Midpoint : **Too High**
3. Check Midpoint : **Too Low**

... *Converges after iterations* ...

**Final High Result:** 

---

### 5. Final Output

When your script finishes, it produces this output:

```text
Testing shot at 4.0m with 12.0m/s...

[SUCCESS] Tuple returned (arc='both' used):
  • Low Angle:  31.25°
  • High Angle: 78.98°

```

* **Low Angle (31.25°):** This is your primary shooting angle. It is fast and efficient.
* **High Angle (78.98°):** This is a very steep "mortar shot." It is technically possible, but the ball would spend a long time in the air, making it harder to aim in a real match.

In [ ]:
result = calculate_shot_angle_optimized(
    v0=test_v0,
    distance=10,
    target_height=2.1,
    coarse_step_deg=1.0,
    arc="both"  # <--- Requesting tuple
)

In [ ]:
# https://firstfrc.blob.core.windows.net/frc2026/FieldAssets/2026-field-dimension-dwgs.pdf
calculate_shot_angle_optimized(
    v0=16.2,
    distance=18.4,
    target_height=0,
    coarse_step_deg=1.0,
    arc="both"  # <--- Requesting tuple
)

(41.0, 42.0)